# Morphometrics Analysis (ADS 2026 — Full QC)

This notebook applies the full quality control exclusion database from the [QC chapter](qc_chapter/index.md) to the corrected ADS 2026 morphometrics. All axons flagged during visual review (border-touching, sub-pixel artifacts, myelin-dominant artifacts, merge errors, invalid g-ratios) are excluded by their specific axon IDs.

## Setup

In [1]:
import pandas as pd
import numpy as np
from scipy import stats as scipy_stats
from pathlib import Path
from IPython.display import display
from PIL import Image
import plotly.graph_objects as go

In [2]:
DATA_DIR = Path("data/macaque/data")
RESULTS_DIR = Path("results")
PLOT_TEMPLATE = "plotly_white"

SAMPLE_SLICES = {
    "sample1": ["01", "03", "07"],
    "sample2": ["01", "03", "05"],
    "sample3": ["01", "03", "05"],
    "sample4": ["01", "05", "07"],
    "sample5": ["01", "03", "05"],
    "sample6": ["01", "03", "05"],
    "sample7": ["01", "03", "05"],
    "sample8": ["01", "03", "05"],
}

REGION_LABELS = {f"sample{i}": f"Region {i}" for i in range(1, 9)}

COLUMN_RENAME = {
    "x0 (px)": "x0", "y0 (px)": "y0",
    "axon_area (um^2)": "axon_area", "axon_perimeter (um)": "axon_perimeter",
    "myelin_area (um^2)": "myelin_area", "axon_diam (um)": "axon_diam",
    "myelin_thickness (um)": "myelin_thickness",
    "axonmyelin_area (um^2)": "axonmyelin_area",
    "axonmyelin_perimeter (um)": "axonmyelin_perimeter",
}

# Load the exclusion database
excl_db = pd.read_csv(RESULTS_DIR / "exclusion_database.csv")
print(f"Exclusion database: {len(excl_db)} axons to exclude")

Exclusion database: 1618 axons to exclude


## Load and filter morphometrics

Each image's morphometrics are loaded from the QC pipeline output (`results/qa_sample*_*/morphometrics.csv`), which uses the same ADS-computed values but preserves the original axon IDs. The exclusion database is then applied to remove all flagged axons by their specific IDs.

In [3]:
all_data = {}
agg_gratio = {}
filter_summary = []

# Ensure slice_id is string for matching
excl_db["slice_id"] = excl_db["slice_id"].astype(str).str.zfill(2)

for sample in sorted(SAMPLE_SLICES.keys()):
    dfs = []
    sample_agg = []
    for slice_id in SAMPLE_SLICES[sample]:
        # Load from QC pipeline output (has original axon IDs as row index)
        qa_folder = RESULTS_DIR / f"qa_{sample}_{slice_id}"
        df = pd.read_csv(qa_folder / "morphometrics.csv")
        df = df.rename(columns=COLUMN_RENAME)
        df = df.drop(columns=[c for c in df.columns if "Unnamed" in str(c)], errors="ignore")
        
        n_before = len(df)
        
        # Get excluded axon IDs for this image
        excl_ids = excl_db[
            (excl_db["sample"] == sample) & (excl_db["slice_id"] == slice_id)
        ]["axon_id"].values
        
        # Exclude by axon ID
        df = df[~df.index.isin(excl_ids)]
        n_after = len(df)
        
        filter_summary.append({
            "Region": REGION_LABELS[sample],
            "Slice": slice_id,
            "Before": n_before,
            "After": n_after,
            "Removed": n_before - n_after,
        })
        
        df["sample"] = sample
        df["slice_id"] = slice_id
        dfs.append(df)

        # Aggregate g-ratio from binary masks
        axon_mask = np.array(Image.open(DATA_DIR / sample / f"{slice_id}_seg-axon.png"))
        myelin_mask = np.array(Image.open(DATA_DIR / sample / f"{slice_id}_seg-myelin.png"))
        avf = np.sum(axon_mask > 0)
        mvf = np.sum(myelin_mask > 0)
        g_agg = np.sqrt(avf / (avf + mvf)) if (avf + mvf) > 0 else np.nan
        sample_agg.append(g_agg)

    combined = pd.concat(dfs, ignore_index=True)
    combined["fiber_diam"] = combined["axon_diam"] + 2 * combined["myelin_thickness"]
    all_data[sample] = combined
    agg_gratio[sample] = sample_agg

display(pd.DataFrame(filter_summary))

,Region,Slice,Before,After,Removed
0,Region 1,01,496,436,60
1,Region 1,03,445,373,72
2,Region 1,07,496,402,94
3,Region 2,01,308,262,46
4,Region 2,03,369,317,52
5,Region 2,05,364,309,55
6,Region 3,01,276,219,57
7,Region 3,03,166,100,66
8,Region 3,05,276,217,59
9,Region 4,01,144,61,83


## Summary statistics

The **aggregate g-ratio** is computed per image from the binary masks using volume fractions: $g_{\text{agg}} = \sqrt{\text{AVF} / \text{FVF}}$ (Stikov et al. 2015), then averaged across images within each region.

In [4]:
summary_rows = []
for sample in sorted(all_data.keys()):
    df = all_data[sample]
    ag = agg_gratio[sample]
    summary_rows.append({
        "Region": REGION_LABELS[sample],
        "n": len(df),
        "Axon diam (\u03bcm)": f"{df['axon_diam'].mean():.4f} \u00b1 {df['axon_diam'].std():.4f}",
        "Fiber diam (\u03bcm)": f"{df['fiber_diam'].mean():.4f} \u00b1 {df['fiber_diam'].std():.4f}",
        "G-ratio (per-axon)": f"{df['gratio'].mean():.4f} \u00b1 {df['gratio'].std():.4f}",
        "G-ratio (aggregate)": f"{np.mean(ag):.4f} \u00b1 {np.std(ag):.4f}",
        "Myelin thickness (\u03bcm)": f"{df['myelin_thickness'].mean():.4f} \u00b1 {df['myelin_thickness'].std():.4f}",
    })

display(pd.DataFrame(summary_rows))

,Region,n,Axon diam (μm),Fiber diam (μm),G-ratio (per-axon),G-ratio (aggregate),Myelin thickness (μm)
0,Region 1,1211,0.7366 ± 0.3165,0.9603 ± 0.3440,0.7522 ± 0.0769,0.7858 ± 0.0213,0.1118 ± 0.0346
1,Region 2,888,0.7642 ± 0.4320,0.9905 ± 0.4793,0.7496 ± 0.0975,0.7991 ± 0.0128,0.1131 ± 0.0481
2,Region 3,536,0.8581 ± 0.6043,1.1036 ± 0.7126,0.7531 ± 0.1011,0.8010 ± 0.0066,0.1228 ± 0.0726
3,Region 4,291,1.0027 ± 0.5436,1.2631 ± 0.6426,0.7764 ± 0.0955,0.8165 ± 0.0167,0.1304 ± 0.0803
4,Region 5,504,0.9556 ± 0.7439,1.2513 ± 0.8410,0.7338 ± 0.1062,0.8106 ± 0.0154,0.1478 ± 0.0778
5,Region 6,646,0.8218 ± 0.5642,1.1627 ± 0.6644,0.6744 ± 0.1172,0.7511 ± 0.0249,0.1705 ± 0.0768
6,Region 7,760,0.7465 ± 0.4994,1.0368 ± 0.5737,0.6881 ± 0.1115,0.7670 ± 0.0121,0.1451 ± 0.0612
7,Region 8,371,0.9531 ± 0.6874,1.2196 ± 0.7611,0.7436 ± 0.1223,0.8309 ± 0.0208,0.1332 ± 0.0755


## Axon diameter vs Fiber diameter

Use the slider to switch between CC regions.

In [5]:
samples_sorted = sorted(all_data.keys())
all_df = pd.concat(all_data.values())
x_max = all_df["axon_diam"].max() * 1.05
y_max = all_df["fiber_diam"].max() * 1.05

fig = go.Figure()

for i, sample in enumerate(samples_sorted):
    df = all_data[sample]
    x, y = df["axon_diam"].values, df["fiber_diam"].values
    visible = (i == 0)

    fig.add_trace(go.Scattergl(
        x=x, y=y, mode="markers",
        marker=dict(size=4, color="steelblue", opacity=0.3),
        showlegend=False, visible=visible,
        hovertemplate="Axon: %{x:.3f} \u03bcm<br>Fiber: %{y:.3f} \u03bcm<extra></extra>",
    ))

    slope, intercept, r_value, _, _ = scipy_stats.linregress(x, y)
    x_line = np.linspace(0, x_max, 100)
    fig.add_trace(go.Scatter(
        x=x_line, y=slope * x_line + intercept, mode="lines",
        line=dict(color="red", width=2), showlegend=False, visible=visible,
        hovertemplate=f"y = {slope:.2f}x + {intercept:.4f}<br>R\u00b2 = {r_value**2:.3f}<extra></extra>",
    ))

n_traces = len(samples_sorted) * 2
steps = []
for i, sample in enumerate(samples_sorted):
    df = all_data[sample]
    x, y = df["axon_diam"].values, df["fiber_diam"].values
    slope, intercept, r_value, _, _ = scipy_stats.linregress(x, y)

    vis = [False] * n_traces
    vis[i * 2] = True
    vis[i * 2 + 1] = True
    step = dict(
        method="update",
        args=[
            {"visible": vis},
            {"title": f"{REGION_LABELS[sample]} (n = {len(df)})",
             "annotations": [
                 dict(text=f"y = {slope:.2f}x + {intercept:.4f}<br>R\u00b2 = {r_value**2:.3f}",
                      xref="paper", yref="paper", x=0.05, y=0.95,
                      xanchor="left", yanchor="top", showarrow=False,
                      font=dict(size=14, color="red")),
             ]},
        ],
        label=REGION_LABELS[sample],
    )
    steps.append(step)

df0 = all_data[samples_sorted[0]]
s0, i0, r0, _, _ = scipy_stats.linregress(df0["axon_diam"].values, df0["fiber_diam"].values)

fig.update_layout(
    sliders=[dict(active=0, steps=steps, currentvalue=dict(prefix="Region: "), pad=dict(t=50))],
    title=f"{REGION_LABELS[samples_sorted[0]]} (n = {len(df0)})",
    xaxis_title="Axon diameter (\u03bcm)",
    yaxis_title="Fiber diameter (\u03bcm)",
    xaxis_range=[0, x_max], yaxis_range=[0, y_max],
    template=PLOT_TEMPLATE, height=600, width=800,
    annotations=[dict(text=f"y = {s0:.2f}x + {i0:.4f}<br>R\u00b2 = {r0**2:.3f}",
                      xref="paper", yref="paper", x=0.05, y=0.95,
                      xanchor="left", yanchor="top", showarrow=False,
                      font=dict(size=14, color="red"))],
)
fig.show()

## G-ratio vs Fiber diameter

Use the slider to switch between CC regions.

In [6]:
x_max_g = all_df["fiber_diam"].max() * 1.05
y_min_g = max(all_df["gratio"].min() * 0.95, 0)
y_max_g = min(all_df["gratio"].max() * 1.05, 1.0)

fig = go.Figure()

for i, sample in enumerate(samples_sorted):
    df = all_data[sample]
    x, y = df["fiber_diam"].values, df["gratio"].values
    visible = (i == 0)

    fig.add_trace(go.Scattergl(
        x=x, y=y, mode="markers",
        marker=dict(size=4, color="darkorange", opacity=0.3),
        showlegend=False, visible=visible,
        hovertemplate="Fiber: %{x:.3f} \u03bcm<br>G-ratio: %{y:.3f}<extra></extra>",
    ))

    slope, intercept, r_value, _, _ = scipy_stats.linregress(x, y)
    x_line = np.linspace(0, x_max_g, 100)
    fig.add_trace(go.Scatter(
        x=x_line, y=slope * x_line + intercept, mode="lines",
        line=dict(color="red", width=2), showlegend=False, visible=visible,
        hovertemplate=f"y = {slope:.4f}x + {intercept:.3f}<br>R\u00b2 = {r_value**2:.3f}<extra></extra>",
    ))

n_traces = len(samples_sorted) * 2
steps = []
for i, sample in enumerate(samples_sorted):
    df = all_data[sample]
    x, y = df["fiber_diam"].values, df["gratio"].values
    slope, intercept, r_value, _, _ = scipy_stats.linregress(x, y)

    vis = [False] * n_traces
    vis[i * 2] = True
    vis[i * 2 + 1] = True
    step = dict(
        method="update",
        args=[
            {"visible": vis},
            {"title": f"{REGION_LABELS[sample]} (n = {len(df)})",
             "annotations": [
                 dict(text=f"y = {slope:.4f}x + {intercept:.3f}<br>R\u00b2 = {r_value**2:.3f}",
                      xref="paper", yref="paper", x=0.05, y=0.05,
                      xanchor="left", yanchor="bottom", showarrow=False,
                      font=dict(size=14, color="red")),
             ]},
        ],
        label=REGION_LABELS[sample],
    )
    steps.append(step)

df0 = all_data[samples_sorted[0]]
s0, i0, r0, _, _ = scipy_stats.linregress(df0["fiber_diam"].values, df0["gratio"].values)

fig.update_layout(
    sliders=[dict(active=0, steps=steps, currentvalue=dict(prefix="Region: "), pad=dict(t=50))],
    title=f"{REGION_LABELS[samples_sorted[0]]} (n = {len(df0)})",
    xaxis_title="Fiber diameter (\u03bcm)",
    yaxis_title="G-ratio",
    xaxis_range=[0, x_max_g], yaxis_range=[y_min_g, y_max_g],
    template=PLOT_TEMPLATE, height=600, width=800,
    annotations=[dict(text=f"y = {s0:.4f}x + {i0:.3f}<br>R\u00b2 = {r0**2:.3f}",
                      xref="paper", yref="paper", x=0.05, y=0.05,
                      xanchor="left", yanchor="bottom", showarrow=False,
                      font=dict(size=14, color="red"))],
)
fig.show()